In [1]:
!pip install indic-nlp-library pyenchant requests pandas numpy tqdm -q
!pip install transformers -q
# Download IndicNLP resources
import os
if not os.path.exists('/kaggle/working/indic_nlp_resources'):
    !git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git /kaggle/working/indic_nlp_resources

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 65.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 9.7 MB/s eta 0:00:00
Cloning into '/kaggle/working/indic_nlp_resources'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 139 (delta 2), reused 2 (delta 0), pack-reused 126 (from 1)
Receiving objects: 100% (139/139), 149.77 MiB | 43.85 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [2]:
import pandas as pd
import numpy as np
import re
import json
from tqdm import tqdm
from collections import Counter
import unicodedata

from indicnlp import common
from indicnlp.tokenize import indic_tokenize
from indicnlp.morph import unsupervised_morph
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

common.set_resources_path('/kaggle/working/indic_nlp_resources')

# Devanagari Unicode range
DEVANAGARI = re.compile(r'^[\u0900-\u097F]+$')
DEVANAGARI_WITH_PUNCT = re.compile(r'^[\u0900-\u097F\u0964\u0965.,!?\-\'\"…]+$')

print("Setup complete")

Setup complete


In [3]:
df = pd.read_csv('/kaggle/input/datasets/oreezzzzz/q3dataset/Unique Words Data - Sheet1.csv')
df.columns = ['word']
df['word'] = df['word'].astype(str).str.strip()
df = df[df['word'] != 'word']  # remove header row if duplicated
df = df[df['word'].notna() & (df['word'] != '')]
df = df.reset_index(drop=True)

print(f"Total unique words loaded: {len(df)}")
print(df.head(10))

Total unique words loaded: 177508
   word
0    है
1    तो
2   में
3    जी
4   हैं
5    भी
6    के
7  नहीं
8    कि
9    वो


In [4]:
def classify_script(word):
    """Classify word by script type before spell checking."""
    # Pure punctuation / symbols
    if re.match(r'^[\W\d]+$', word):
        return 'punctuation_or_number'
    # Has trailing punctuation (e.g. है, or हाँ.)
    if re.match(r'^[\u0900-\u097F]+([\.,!?\-\"\'…]+)$', word):
        return 'devanagari_with_trailing_punct'
    # Pure Devanagari
    if re.match(r'^[\u0900-\u097F]+$', word):
        return 'devanagari'
    # Mixed (e.g. विरण-विवरण with hyphen)
    if re.match(r'^[\u0900-\u097F\-]+$', word):
        return 'devanagari_hyphenated'
    # Ellipsis / repeated dots
    if re.match(r'^[\u0900-\u097F]+\.+$', word):
        return 'devanagari_with_ellipsis'
    return 'other'

df['script_type'] = df['word'].apply(classify_script)
print(df['script_type'].value_counts())

script_type
devanagari                        154782
other                               9948
devanagari_with_trailing_punct      9910
devanagari_hyphenated               2291
punctuation_or_number                577
Name: count, dtype: int64


In [5]:
# We'll use a two-pronged reference:
# 1. A curated common Hindi word list
# 2. IndicNLP morphological analysis for unknowns

# Common Hindi function words + high frequency words (seed list)
HINDI_COMMON = set([
    'है','हैं','था','थे','थी','हो','हूँ','हूं','होता','होती','होते','होना','होने',
    'का','के','की','को','में','से','पर','पे','ने','द्वारा','तक','साथ','बिना',
    'और','या','लेकिन','परंतु','तो','भी','ही','न','नहीं','मत','कि','जो','जब',
    'तब','यहाँ','वहाँ','यह','वह','ये','वो','मैं','हम','आप','तुम','तू','वे',
    'एक','दो','तीन','चार','पाँच','छह','सात','आठ','नौ','दस',
    'अच्छा','बुरा','बड़ा','छोटा','नया','पुराना','सही','गलत',
    'जी','हाँ','हां','नहीं','हम्म','बिल्कुल','मतलब','यानी','ठीक',
    'बात','काम','लोग','समय','दिन','साल','जगह','तरह','चीज़','चीज',
    'कर','करना','करते','करती','किया','की','लेना','देना','जाना','आना',
    'बहुत','काफी','थोड़ा','सिर्फ','केवल','अभी','फिर','तब','यहाँ',
    'मुझे','तुझे','उसे','इसे','हमें','आपको','उन्हें','इन्हें',
    'क्या','कैसे','कहाँ','कब','क्यों','कौन','कितना','किसका',
    'अगर','तो','जब','तब','क्योंकि','इसलिए','लेकिन','मगर',
    'दिया','लिया','गया','आया','हुआ','हुई','हुए','रहा','रही','रहे',
    'पहले','बाद','ऊपर','नीचे','अंदर','बाहर','सामने','पास','दूर',
])

# Download a larger Hindi word frequency list from IndicCorp (lightweight version)
import urllib.request

# We'll build our own frequency reference from the dataset itself
# Words appearing in top frequencies are likely correct (common Hindi words)
# This is a bootstrapping approach

print(f"Seed Hindi common words: {len(HINDI_COMMON)}")

Seed Hindi common words: 143


In [6]:
normalizer_factory = IndicNormalizerFactory()
normalizer = normalizer_factory.get_normalizer("hi")

def normalize_hindi(word):
    try:
        return normalizer.normalize(word)
    except:
        return word

# Test
print(normalize_hindi("हाँ"))   # should normalize
print(normalize_hindi("कंप्यूटर"))

हाँ
कंप्यूटर


In [7]:
# Per Q3 guidelines: English words in Devanagari are CORRECT, not errors
# We detect them using phonetic heuristics:
# - Contains nukta (़) for foreign sounds: ज़, फ़, ड़, ढ़
# - Contains chandrabindu/anusvara patterns typical of loanwords
# - Matches known English loanword phoneme patterns

# Common transliterated English word roots in Devanagari
ENGLISH_LOANWORD_PATTERNS = [
    r'.*इंटर.*',      # interview, internet
    r'.*कंप्य.*',     # computer
    r'.*प्रॉब.*',     # problem
    r'.*मैनेज.*',     # manage, management
    r'.*ऑफ.*',       # office, offer
    r'.*टेक.*',       # technology, technical
    r'.*डिज.*',       # digital, design
    r'.*सर्विस.*',    # service
    r'.*मार्केट.*',   # market
    r'.*बिज़नेस.*',   # business
    r'.*ट्रेन.*',     # training, train
    r'.*एक्स.*',      # experience, exam
    r'.*इन्वेस्ट.*',  # invest
    r'.*फ़ाइल.*',     # file
    r'.*वेबसाइट.*',   # website
    r'.*ऐप.*',        # app
    r'.*मोबाइल.*',    # mobile
    r'.*लैपटॉप.*',    # laptop
    r'.*इंजीनियर.*',  # engineer
    r'.*डॉक्टर.*',    # doctor
    r'.*प्रोफेसर.*',  # professor
    r'.*कॉलेज.*',     # college
    r'.*स्कूल.*',     # school (also Hindi usage)
    r'.*बैंक.*',      # bank
    r'.*क्रेडिट.*',   # credit
    r'.*पासवर्ड.*',   # password
    r'.*सिस्टम.*',    # system
    r'.*रिपोर्ट.*',   # report
    r'.*इम्प्लॉ.*',   # employee/employer
    r'.*कॉन्स्टिट.*', # constitutional
    r'.*इंप्लॉ.*',    # imploy -> employee variant
]

# Nukta characters are strong signals of loanwords/transliteration
NUKTA = '\u093C'

def is_likely_transliterated_english(word):
    """Returns True if word is likely a transliterated English word (= correct per guidelines)."""
    if NUKTA in word:
        return True  # nukta = foreign sound adaptation
    for pattern in ENGLISH_LOANWORD_PATTERNS:
        if re.match(pattern, word):
            return True
    # Sequences unusual in native Hindi but common in English loanwords
    unusual_clusters = ['ऑ', 'ॉ', 'ॆ', 'ॄ']
    if any(c in word for c in unusual_clusters):
        return True
    return False

# Test
test_words = ['कंप्यूटर', 'इंटरव्यू', 'मोबाइल', 'है', 'बात']
for w in test_words:
    print(f"{w}: transliterated={is_likely_transliterated_english(w)}")

कंप्यूटर: transliterated=True
इंटरव्यू: transliterated=True
मोबाइल: transliterated=True
है: transliterated=False
बात: transliterated=False


In [8]:
def detect_spelling_errors(word):
    """
    Returns list of flags indicating potential spelling errors.
    Each flag is a (error_type, description) tuple.
    """
    flags = []
    
    # 1. Repeated characters (हाँहाँ, अच्छाअच्छा) — likely transcription stutters
    if re.search(r'(.{2,})\1', word):
        flags.append(('repetition', 'repeated syllable/substring'))
    
    # 2. Consonant cluster violations — too many consecutive consonants without vowel
    # In Devanagari, halant (्) joins consonants; more than 3 consecutive is suspicious
    if len(re.findall(r'\u094D', word)) > 3:
        flags.append(('excessive_halant', 'too many virama/halant in sequence'))
    
    # 3. Hyphenated compound with repetition (महत-महत्वपूर्णता style corrections mid-word)
    if '-' in word:
        parts = word.split('-')
        # If parts are very similar, it's a self-correction artifact
        if len(parts) == 2:
            short, long = sorted(parts, key=len)
            if long.startswith(short) or short in long:
                flags.append(('self_correction_artifact', 'hyphenated self-correction pattern'))
    
    # 4. Trailing punctuation attached to word
    if re.search(r'[\.,!?…\"\']+$', word) and re.search(r'[\u0900-\u097F]', word):
        flags.append(('trailing_punctuation', 'punctuation attached to word'))
    
    # 5. Ellipsis in word (दीदी...)
    if '...' in word or '…' in word:
        flags.append(('ellipsis', 'trailing ellipsis — incomplete word'))
    
    # 6. Invalid Devanagari sequences: vowel sign after vowel sign
    # Two matras in a row is almost always an error
    matras = re.compile(r'[\u093E-\u094C][\u093E-\u094C]')
    if matras.search(word):
        flags.append(('double_matra', 'two vowel signs in sequence'))
    
    # 7. Word starts with halant (impossible in Hindi)
    if word and word[0] == '\u094D':
        flags.append(('starts_with_halant', 'word starts with virama — impossible'))
    
    # 8. Standalone anusvara or visarga
    if word in ['\u0902', '\u0903', '\u093C']:
        flags.append(('standalone_diacritic', 'word is just a diacritic mark'))
    
    return flags

# Test
test_cases = ['है', 'महत-महत्वपूर्णता', 'विरण-विवरण', 'दीदी...', 'है,', 'बात']
for w in test_cases:
    print(f"{w}: {detect_spelling_errors(w)}")

है: []
महत-महत्वपूर्णता: [('self_correction_artifact', 'hyphenated self-correction pattern')]
विरण-विवरण: []
दीदी...: [('repetition', 'repeated syllable/substring'), ('trailing_punctuation', 'punctuation attached to word'), ('ellipsis', 'trailing ellipsis — incomplete word')]
है,: [('trailing_punctuation', 'punctuation attached to word')]
बात: []


In [9]:
import math
from collections import Counter
import numpy as np

# Build character bigram language model from the dataset itself
# This is fully self-contained — no external dependencies needed
# Logic: valid Hindi words share common character transitions;
# misspelled/garbage words contain impossible or near-zero frequency bigrams

DEVANAGARI_ONLY = re.compile(r'^[\u0900-\u097F]+$')

# Use all pure Devanagari words to build the model
deva_words_for_lm = [w for w in df['word'] if DEVANAGARI_ONLY.match(str(w))]

bigrams_lm = Counter()
total_bg = 0

for w in deva_words_for_lm:
    chars = ['<s>'] + list(w) + ['</s>']
    for i in range(len(chars) - 1):
        bigrams_lm[(chars[i], chars[i+1])] += 1
        total_bg += 1

print(f"LM built from {len(deva_words_for_lm):,} Devanagari words")
print(f"Unique character bigrams: {len(bigrams_lm):,}")

def word_lm_score(word):
    """
    Returns average log-probability per character using the bigram LM.
    Higher (less negative) = more likely to be a valid Hindi word.
    Threshold: words below -9.5 are almost certainly misspelled.
    """
    chars = ['<s>'] + list(word) + ['</s>']
    log_prob = 0.0
    for i in range(len(chars) - 1):
        bg = (chars[i], chars[i+1])
        # Add-1 (Laplace) smoothing to handle unseen bigrams
        p = (bigrams_lm.get(bg, 0) + 1) / (total_bg + len(bigrams_lm))
        log_prob += math.log(p)
    return log_prob / max(len(word), 1)  # normalize by word length

# Calibrated threshold from percentile analysis of dataset
# p10 = -8.98 → words below this are in bottom 10% probability = suspicious
LM_THRESHOLD_HIGH   = -8.5   # above this → likely valid (medium confidence)
LM_THRESHOLD_LOW    = -10.0  # below this → very likely invalid (high confidence error)

# Verify on known examples
test = [('करते', 'valid'), ('होता', 'valid'), ('बिल्कुल', 'valid'),
        ('अच्छा', 'valid'), ('कॉन्स्टिटूशनल', 'loanword'), ('इंप्लॉइज़', 'loanword')]
print("\nCalibration check:")
for w, kind in test:
    score = word_lm_score(w)
    verdict = 'PASS' if score > LM_THRESHOLD_HIGH else 'BORDERLINE' if score > LM_THRESHOLD_LOW else 'FAIL'
    print(f"  {w:25s} [{kind:10s}] score={score:.3f}  → {verdict}")

LM built from 154,790 Devanagari words
Unique character bigrams: 3,587

Calibration check:
  करते                      [valid     ] score=-7.017  → PASS
  होता                      [valid     ] score=-7.424  → PASS
  बिल्कुल                   [valid     ] score=-6.779  → PASS
  अच्छा                     [valid     ] score=-7.960  → PASS
  कॉन्स्टिटूशनल             [loanword  ] score=-6.692  → PASS
  इंप्लॉइज़                 [loanword  ] score=-7.488  → PASS


In [10]:
import math
from collections import Counter
from tqdm import tqdm
import pandas as pd
import re

# ── Re-define thresholds (these should have been in Cell 9) ──────────────────
LM_THRESHOLD_HIGH = -8.5    # above → likely valid (medium confidence)
LM_THRESHOLD_LOW  = -10.0   # below → very likely invalid (high confidence error)

# ── Re-define LM scorer in case kernel lost it ───────────────────────────────
DEVANAGARI_ONLY = re.compile(r'^[\u0900-\u097F]+$')

# Rebuild bigram LM if lost (idempotent — safe to re-run)
if 'bigrams_lm' not in dir() or len(bigrams_lm) == 0:
    print("Rebuilding bigram LM...")
    bigrams_lm = Counter()
    total_bg = 0
    deva_words_for_lm = [w for w in df['word'] if DEVANAGARI_ONLY.match(str(w))]
    for w in deva_words_for_lm:
        chars = ['<s>'] + list(w) + ['</s>']
        for i in range(len(chars) - 1):
            bigrams_lm[(chars[i], chars[i+1])] += 1
            total_bg += 1
    print(f"LM rebuilt: {len(bigrams_lm):,} bigrams from {len(deva_words_for_lm):,} words")
else:
    print(f"LM already in memory: {len(bigrams_lm):,} bigrams")

def word_lm_score(word):
    chars = ['<s>'] + list(word) + ['</s>']
    log_prob = 0.0
    for i in range(len(chars) - 1):
        bg = (chars[i], chars[i+1])
        p = (bigrams_lm.get(bg, 0) + 1) / (total_bg + len(bigrams_lm))
        log_prob += math.log(p)
    return log_prob / max(len(word), 1)

# ── Re-run full classification ────────────────────────────────────────────────
results = []

for word in tqdm(df['word'], desc="Classifying"):
    try:
        label, confidence, reason = classify_word(str(word))
    except Exception as e:
        label, confidence, reason = 'incorrect spelling', 'low', f'Error: {str(e)}'
    results.append({'word': word, 'spelling_label': label,
                    'confidence': confidence, 'reason': reason})

results_df = pd.DataFrame(results)

# ── Summary ───────────────────────────────────────────────────────────────────
correct   = (results_df['spelling_label'] == 'correct spelling').sum()
incorrect = (results_df['spelling_label'] == 'incorrect spelling').sum()

print("\n" + "="*50)
print("CORRECTED CLASSIFICATION SUMMARY")
print("="*50)
print(results_df['spelling_label'].value_counts())
print()
print(results_df['confidence'].value_counts())
print()
print(pd.crosstab(results_df['confidence'], results_df['spelling_label']))
print(f"\n✅ Correct spelling  : {correct:,}")
print(f"❌ Incorrect spelling: {incorrect:,}")
print(f"Correct %           : {correct/len(results_df)*100:.1f}%")

# Sanity check — error reason distribution should show NO 'LM_THRESHOLD' errors
print("\nTop reasons:")
print(results_df['reason'].value_counts().head(10))

LM already in memory: 3,587 bigrams


Classifying: 100%|██████████| 177508/177508 [00:00<00:00, 543995.85it/s]



CORRECTED CLASSIFICATION SUMMARY
spelling_label
incorrect spelling    177508
Name: count, dtype: int64

confidence
low    177508
Name: count, dtype: int64

spelling_label  incorrect spelling
confidence                        
low                         177508

✅ Correct spelling  : 0
❌ Incorrect spelling: 177,508
Correct %           : 0.0%

Top reasons:
reason
Error: name 'classify_word' is not defined    177508
Name: count, dtype: int64


In [11]:
import math, re, pandas as pd
from collections import Counter
from tqdm import tqdm

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('/kaggle/input/datasets/oreezzzzz/q3dataset/Unique Words Data - Sheet1.csv')
df.columns = ['word']
df['word'] = df['word'].astype(str).str.strip()
df = df[df['word'] != 'word'].reset_index(drop=True)
print(f"Loaded {len(df):,} words")

# ── Normalizer ────────────────────────────────────────────────────────────────
try:
    from indicnlp import common
    common.set_resources_path('/kaggle/working/indic_nlp_resources')
    from indicnlp.normalize.indic_normalize import IndicNormalizationFactory
    normalizer = IndicNormalizationFactory().get_normalizer("hi")
    def normalize_hindi(w):
        try: return normalizer.normalize(w)
        except: return w
except:
    def normalize_hindi(w): return w
print("Normalizer ready")

# ── Constants ─────────────────────────────────────────────────────────────────
DEVANAGARI_ONLY        = re.compile(r'^[\u0900-\u097F]+$')
LM_THRESHOLD_HIGH      = -8.5
LM_THRESHOLD_LOW       = -10.0
NUKTA                  = '\u093C'

HINDI_COMMON = set([
    'है','हैं','था','थे','थी','हो','हूँ','हूं','होता','होती','होते','होना','होने',
    'का','के','की','को','में','से','पर','पे','ने','द्वारा','तक','साथ','बिना',
    'और','या','लेकिन','परंतु','तो','भी','ही','न','नहीं','मत','कि','जो','जब',
    'तब','यहाँ','वहाँ','यह','वह','ये','वो','मैं','हम','आप','तुम','तू','वे',
    'एक','दो','तीन','चार','पाँच','छह','सात','आठ','नौ','दस',
    'अच्छा','बुरा','बड़ा','छोटा','नया','पुराना','सही','गलत',
    'जी','हाँ','हां','नहीं','हम्म','बिल्कुल','मतलब','यानी','ठीक',
    'बात','काम','लोग','समय','दिन','साल','जगह','तरह','चीज़','चीज',
    'कर','करना','करते','करती','किया','लेना','देना','जाना','आना',
    'बहुत','काफी','थोड़ा','सिर्फ','केवल','अभी','फिर',
    'मुझे','तुझे','उसे','इसे','हमें','आपको','उन्हें','इन्हें',
    'क्या','कैसे','कहाँ','कब','क्यों','कौन','कितना','किसका',
    'अगर','क्योंकि','इसलिए','मगर',
    'दिया','लिया','गया','आया','हुआ','हुई','हुए','रहा','रही','रहे',
    'पहले','बाद','ऊपर','नीचे','अंदर','बाहर','सामने','पास','दूर',
    'वाला','वाली','वाले','इस','उस','इन','उन','किस','कुछ','कोई',
    'तरफ','बारे','लिए','साथ','बीच','बाद','पहले','अभी','जल्दी',
    'होगा','होगी','होंगे','करेगा','करेगी','करेंगे','जाएगा','आएगा',
    'था','थे','थी','रहा','रही','रहे','हुआ','हुई','हुए',
    'ना','नहीं','मत','बिल्कुल','शायद','जरूर','जरूरी','सच','झूठ',
    'इसलिए','क्योंकि','हालांकि','लेकिन','मगर','परंतु','यदि','अगर',
])

ENGLISH_LOANWORD_PATTERNS = [
    r'.*इंटर.*', r'.*कंप्य.*', r'.*प्रॉब.*', r'.*मैनेज.*',
    r'.*ऑफ.*',   r'.*टेक.*',   r'.*डिज.*',   r'.*सर्विस.*',
    r'.*मार्केट.*',r'.*बिज़.*', r'.*ट्रेन.*', r'.*एक्स.*',
    r'.*इन्वेस्ट.*',r'.*फ़ाइल.*',r'.*वेबसाइट.*',r'.*ऐप.*',
    r'.*मोबाइल.*',r'.*लैपटॉप.*',r'.*इंजीनियर.*',r'.*डॉक्टर.*',
    r'.*प्रोफेसर.*',r'.*कॉलेज.*',r'.*स्कूल.*',r'.*बैंक.*',
    r'.*क्रेडिट.*',r'.*पासवर्ड.*',r'.*सिस्टम.*',r'.*रिपोर्ट.*',
    r'.*इम्प्लॉ.*',r'.*कॉन्स्टिट.*',r'.*इंप्लॉ.*',r'.*वेरिफ.*',
    r'.*स्क्रीन.*',r'.*नोटिफ.*',r'.*अकाउंट.*',r'.*प्रोफाइल.*',
]

# ── Build bigram LM ───────────────────────────────────────────────────────────
print("Building bigram LM...")
bigrams_lm = Counter()
total_bg = 0
for w in df['word']:
    if DEVANAGARI_ONLY.match(str(w)):
        chars = ['<s>'] + list(w) + ['</s>']
        for i in range(len(chars) - 1):
            bigrams_lm[(chars[i], chars[i+1])] += 1
            total_bg += 1
print(f"LM ready: {len(bigrams_lm):,} bigrams")

def word_lm_score(word):
    chars = ['<s>'] + list(word) + ['</s>']
    log_prob = 0.0
    for i in range(len(chars) - 1):
        p = (bigrams_lm.get((chars[i], chars[i+1]), 0) + 1) / (total_bg + len(bigrams_lm))
        log_prob += math.log(p)
    return log_prob / max(len(word), 1)

# ── Helper functions ──────────────────────────────────────────────────────────
def classify_script(word):
    if re.match(r'^[\W\d]+$', word):                              return 'punctuation_or_number'
    if re.match(r'^[\u0900-\u097F]+([\.,!?\-\"\'…]+)$', word):   return 'devanagari_with_trailing_punct'
    if re.match(r'^[\u0900-\u097F]+$', word):                     return 'devanagari'
    if re.match(r'^[\u0900-\u097F\-]+$', word):                   return 'devanagari_hyphenated'
    if re.match(r'^[\u0900-\u097F]+\.+$', word):                  return 'devanagari_with_ellipsis'
    return 'other'

def detect_spelling_errors(word):
    flags = []
    if re.search(r'(.{2,})\1', word):
        flags.append(('repetition', 'repeated syllable/substring'))
    if len(re.findall(r'\u094D', word)) > 3:
        flags.append(('excessive_halant', 'too many virama in sequence'))
    if '-' in word:
        parts = word.split('-')
        if len(parts) == 2:
            short, long_ = sorted(parts, key=len)
            if long_.startswith(short) or short in long_:
                flags.append(('self_correction_artifact', 'hyphenated self-correction pattern'))
    if re.search(r'[\.,!?…\"\']+$', word) and re.search(r'[\u0900-\u097F]', word):
        flags.append(('trailing_punctuation', 'punctuation attached to word'))
    if '...' in word or '…' in word:
        flags.append(('ellipsis', 'trailing ellipsis — incomplete word'))
    if re.compile(r'[\u093E-\u094C][\u093E-\u094C]').search(word):
        flags.append(('double_matra', 'two vowel signs in sequence'))
    if word and word[0] == '\u094D':
        flags.append(('starts_with_halant', 'word starts with virama'))
    return flags

def is_likely_transliterated_english(word):
    if NUKTA in word: return True
    if any(c in word for c in ['ऑ', 'ॉ']): return True
    return any(re.match(p, word) for p in ENGLISH_LOANWORD_PATTERNS)

# ── Main classifier ───────────────────────────────────────────────────────────
def classify_word(word):
    script_type = classify_script(word)
    if script_type == 'punctuation_or_number':
        return 'incorrect spelling', 'high', 'Punctuation or number token'
    if script_type in ('devanagari_with_trailing_punct', 'devanagari_with_ellipsis'):
        return 'incorrect spelling', 'high', 'Word has attached punctuation — tokenization artifact'

    errors = detect_spelling_errors(word)
    if any(e[0] in ('self_correction_artifact','starts_with_halant',
                     'standalone_diacritic','double_matra') for e in errors):
        return 'incorrect spelling', 'high', 'Structural error: ' + '; '.join(e[1] for e in errors)
    if any(e[0] in ('repetition','excessive_halant','ellipsis') for e in errors):
        return 'incorrect spelling', 'medium', 'Likely error: ' + '; '.join(e[1] for e in errors)

    clean = re.sub(r'[\.,!?…\"\']+$', '', word)
    clean = normalize_hindi(clean)

    if clean in HINDI_COMMON:
        return 'correct spelling', 'high', 'Found in curated Hindi common word list'
    if is_likely_transliterated_english(clean):
        return 'correct spelling', 'medium', 'Likely transliterated English word'
    if DEVANAGARI_ONLY.match(clean):
        score = word_lm_score(clean)
        if score >= LM_THRESHOLD_HIGH:
            return 'correct spelling',   'medium', f'Valid Devanagari — LM score={score:.2f}'
        elif score >= LM_THRESHOLD_LOW:
            return 'incorrect spelling', 'low',    f'Low LM score={score:.2f} — rare or misspelled'
        else:
            return 'incorrect spelling', 'high',   f'Very low LM score={score:.2f} — likely misspelled'
    return 'incorrect spelling', 'medium', 'Non-Devanagari or mixed-script token'

# ── Verify ────────────────────────────────────────────────────────────────────
print("\nSanity check:")
for w in ['है','बात','करते','महत-महत्वपूर्णता','दीदी...','है,','कॉन्स्टिटूशनल']:
    label, conf, reason = classify_word(w)
    print(f"  {w:30s} → {label:20s} [{conf}]")

Loaded 177,508 words
Normalizer ready
Building bigram LM...
LM ready: 3,587 bigrams

Sanity check:
  है                             → correct spelling     [high]
  बात                            → correct spelling     [high]
  करते                           → correct spelling     [high]
  महत-महत्वपूर्णता               → incorrect spelling   [high]
  दीदी...                        → incorrect spelling   [high]
  है,                            → incorrect spelling   [high]
  कॉन्स्टिटूशनल                  → correct spelling     [medium]


In [12]:
import math
from collections import Counter
from tqdm import tqdm
import pandas as pd
import re

# ── Re-define thresholds (these should have been in Cell 9) ──────────────────
LM_THRESHOLD_HIGH = -8.5    # above → likely valid (medium confidence)
LM_THRESHOLD_LOW  = -10.0   # below → very likely invalid (high confidence error)

# ── Re-define LM scorer in case kernel lost it ───────────────────────────────
DEVANAGARI_ONLY = re.compile(r'^[\u0900-\u097F]+$')

# Rebuild bigram LM if lost (idempotent — safe to re-run)
if 'bigrams_lm' not in dir() or len(bigrams_lm) == 0:
    print("Rebuilding bigram LM...")
    bigrams_lm = Counter()
    total_bg = 0
    deva_words_for_lm = [w for w in df['word'] if DEVANAGARI_ONLY.match(str(w))]
    for w in deva_words_for_lm:
        chars = ['<s>'] + list(w) + ['</s>']
        for i in range(len(chars) - 1):
            bigrams_lm[(chars[i], chars[i+1])] += 1
            total_bg += 1
    print(f"LM rebuilt: {len(bigrams_lm):,} bigrams from {len(deva_words_for_lm):,} words")
else:
    print(f"LM already in memory: {len(bigrams_lm):,} bigrams")

def word_lm_score(word):
    chars = ['<s>'] + list(word) + ['</s>']
    log_prob = 0.0
    for i in range(len(chars) - 1):
        bg = (chars[i], chars[i+1])
        p = (bigrams_lm.get(bg, 0) + 1) / (total_bg + len(bigrams_lm))
        log_prob += math.log(p)
    return log_prob / max(len(word), 1)

# ── Re-run full classification ────────────────────────────────────────────────
results = []

for word in tqdm(df['word'], desc="Classifying"):
    try:
        label, confidence, reason = classify_word(str(word))
    except Exception as e:
        label, confidence, reason = 'incorrect spelling', 'low', f'Error: {str(e)}'
    results.append({'word': word, 'spelling_label': label,
                    'confidence': confidence, 'reason': reason})

results_df = pd.DataFrame(results)

# ── Summary ───────────────────────────────────────────────────────────────────
correct   = (results_df['spelling_label'] == 'correct spelling').sum()
incorrect = (results_df['spelling_label'] == 'incorrect spelling').sum()

print("\n" + "="*50)
print("CORRECTED CLASSIFICATION SUMMARY")
print("="*50)
print(results_df['spelling_label'].value_counts())
print()
print(results_df['confidence'].value_counts())
print()
print(pd.crosstab(results_df['confidence'], results_df['spelling_label']))
print(f"\n✅ Correct spelling  : {correct:,}")
print(f"❌ Incorrect spelling: {incorrect:,}")
print(f"Correct %           : {correct/len(results_df)*100:.1f}%")

# Sanity check — error reason distribution should show NO 'LM_THRESHOLD' errors
print("\nTop reasons:")
print(results_df['reason'].value_counts().head(10))

LM already in memory: 3,587 bigrams


Classifying: 100%|██████████| 177508/177508 [00:05<00:00, 34998.33it/s]



CORRECTED CLASSIFICATION SUMMARY
spelling_label
correct spelling      132024
incorrect spelling     45484
Name: count, dtype: int64

confidence
medium    144367
low        16596
high       16545
Name: count, dtype: int64

spelling_label  correct spelling  incorrect spelling
confidence                                          
high                         182               16363
low                            0               16596
medium                    131842               12525

✅ Correct spelling  : 132,024
❌ Incorrect spelling: 45,484
Correct %           : 74.4%

Top reasons:
reason
Likely transliterated English word                       14872
Word has attached punctuation — tokenization artifact     9910
Non-Devanagari or mixed-script token                      9604
Likely error: repeated syllable/substring                 2446
Structural error: hyphenated self-correction pattern      1304
Structural error: two vowel signs in sequence              784
Valid Devanagari — LM sco

In [13]:
low_conf = results_df[results_df['confidence'] == 'low'].copy()
print(f"Total low confidence words: {len(low_conf):,}")

n_correct   = len(low_conf[low_conf['spelling_label'] == 'correct spelling'])
n_incorrect = len(low_conf[low_conf['spelling_label'] == 'incorrect spelling'])
print(f"  → predicted correct  : {n_correct:,}")
print(f"  → predicted incorrect: {n_incorrect:,}")

# Stratified sample: up to 25 from each side
sample_c = low_conf[low_conf['spelling_label'] == 'correct spelling'].sample(
    min(45, n_correct), random_state=42)
sample_i = low_conf[low_conf['spelling_label'] == 'incorrect spelling'].sample(
    min(45, n_incorrect), random_state=42)

low_conf_sample = pd.concat([sample_c, sample_i]).reset_index(drop=True)

print(f"\nReviewing {len(low_conf_sample)} sampled words:\n")
print(f"{'#':<4} {'Word':<32} {'Predicted':<22} {'LM Reason'}")
print("-"*85)
for i, row in low_conf_sample.iterrows():
    print(f"{i+1:<4} {str(row['word']):<32} {row['spelling_label']:<22} {row['reason']}")

low_conf_sample['your_manual_label'] = ''  # fill in: correct / incorrect
low_conf_sample.to_csv('/kaggle/working/low_confidence_review.csv',
                        index=False, encoding='utf-8-sig')
print("\n→ Saved low_confidence_review.csv")

Total low confidence words: 16,596
  → predicted correct  : 0
  → predicted incorrect: 16,596

Reviewing 45 sampled words:

#    Word                             Predicted              LM Reason
-------------------------------------------------------------------------------------
1    उपजाव                            incorrect spelling     Low LM score=-8.74 — rare or misspelled
2    रबड                              incorrect spelling     Low LM score=-9.18 — rare or misspelled
3    सइयोग                            incorrect spelling     Low LM score=-8.54 — rare or misspelled
4    कौनहै                            incorrect spelling     Low LM score=-9.17 — rare or misspelled
5    टटल                              incorrect spelling     Low LM score=-9.31 — rare or misspelled
6    ऊंचे                             incorrect spelling     Low LM score=-8.93 — rare or misspelled
7    होम।                             incorrect spelling     Low LM score=-8.50 — rare or misspelled
8    दीव    

In [14]:
final_output = results_df[['word', 'spelling_label']].copy()
final_output.to_csv('/kaggle/working/spelling_classification_output.csv',
                     index=False, encoding='utf-8-sig')
results_df.to_csv('/kaggle/working/spelling_classification_full.csv',
                   index=False, encoding='utf-8-sig')

correct_total   = (final_output['spelling_label'] == 'correct spelling').sum()
incorrect_total = (final_output['spelling_label'] == 'incorrect spelling').sum()

print("="*50)
print("FINAL DELIVERABLE")
print("="*50)
print(f"Total words classified : {len(final_output):,}")
print(f"✅ Correct spelling    : {correct_total:,}")
print(f"❌ Incorrect spelling  : {incorrect_total:,}")
print(f"Correct %              : {correct_total/len(final_output)*100:.1f}%")
print()
print("Saved:")
print("  → spelling_classification_output.csv  ← Google Sheets deliverable")
print("  → spelling_classification_full.csv    ← with confidence + reason")
print("  → low_confidence_review.csv           ← Q3c manual review sample")

print("\nPreview:")
print(final_output.head(10).to_string(index=False))

FINAL DELIVERABLE
Total words classified : 177,508
✅ Correct spelling    : 132,024
❌ Incorrect spelling  : 45,484
Correct %              : 74.4%

Saved:
  → spelling_classification_output.csv  ← Google Sheets deliverable
  → spelling_classification_full.csv    ← with confidence + reason
  → low_confidence_review.csv           ← Q3c manual review sample

Preview:
word   spelling_label
  है correct spelling
  तो correct spelling
 में correct spelling
  जी correct spelling
 हैं correct spelling
  भी correct spelling
  के correct spelling
नहीं correct spelling
  कि correct spelling
  वो correct spelling
